# 2026 NET_SALES Forecast — Linear Regression per Series (Snowpark input)

For every `PARENT_DEALER_CODE_MODEL_FAMILY` x calendar day (Sep-01 to Dec-10), we have up to
3 historical `NET_SALES` values — one per year (2023, 2024, 2025). We fit

    NET_SALES = slope * YEAR + intercept

across those points and extrapolate to 2026.

This version takes a **Snowpark DataFrame** (`df`) as input instead of reading a CSV. It converts
to pandas once, then runs the same fast vectorized closed-form OLS as before (no per-group
`sklearn` loop — with ~31,893 series x ~101 days ≈ 3.2M tiny regressions, that would be far too
slow).

**Edge cases handled:**
- 3 valid years → standard least-squares line
- 2 valid years → line through the 2 points
- 1 valid year → flat-line forecast = that single value
- 0 valid years → left as `NaN`
- Negative predictions clipped to 0
- Column names are normalized to uppercase (Snowflake often returns them that way already, but this makes it robust either way)
- `CAL_DATE` is parsed with `dayfirst=True`, so it works whether Snowpark hands you a string like `"01-09-2023"` or a native `DATE`/`TIMESTAMP` column

In [ ]:
import numpy as np
import pandas as pd

# If you need to open a Snowpark session yourself in this notebook, uncomment:
# from snowflake.snowpark import Session
# session = Session.builder.configs(connection_parameters).create()
# df = session.table("YOUR_SCHEMA.YOUR_TABLE")

# Otherwise, assume `df` is already a Snowpark DataFrame available in scope,
# e.g. produced by an earlier cell or passed in from your pipeline.

## 1. Config

In [ ]:
HIST_YEARS   = [2023, 2024, 2025]
PREDICT_YEAR = 2026
WINDOW_START = "09-01"   # MM-DD
WINDOW_END   = "12-10"   # MM-DD

## 2. Core functions

In [ ]:
def normalize_columns(pdf: pd.DataFrame) -> pd.DataFrame:
    """Uppercase + strip column names, and confirm the required columns exist."""
    pdf = pdf.copy()
    pdf.columns = [c.strip().upper() for c in pdf.columns]
    required = {"PARENT_DEALER_CODE_MODEL_FAMILY", "CAL_DATE", "NET_SALES"}
    missing = required - set(pdf.columns)
    if missing:
        raise ValueError(f"Missing required columns after normalization: {missing}")
    return pdf


def prep_dates(pdf: pd.DataFrame) -> pd.DataFrame:
    """Parse CAL_DATE (string 'DD-MM-YYYY' OR native date/timestamp) and derive YEAR / MONTH_DAY."""
    pdf = pdf.copy()
    pdf["CAL_DATE_PARSED"] = pd.to_datetime(pdf["CAL_DATE"], dayfirst=True, errors="coerce")

    n_bad = pdf["CAL_DATE_PARSED"].isna().sum()
    if n_bad:
        print(f"[WARN] {n_bad} rows had unparseable CAL_DATE and will be dropped.")
        pdf = pdf.dropna(subset=["CAL_DATE_PARSED"])

    pdf["YEAR"] = pdf["CAL_DATE_PARSED"].dt.year
    pdf["MONTH_DAY"] = pdf["CAL_DATE_PARSED"].dt.strftime("%m-%d")
    pdf["NET_SALES"] = pd.to_numeric(pdf["NET_SALES"], errors="coerce")
    return pdf


def build_year_matrix(pdf: pd.DataFrame, hist_years) -> pd.DataFrame:
    """One row per (series, month-day), one column per historical year."""
    grouped = (
        pdf.groupby(["PARENT_DEALER_CODE_MODEL_FAMILY", "MONTH_DAY", "YEAR"])["NET_SALES"]
        .mean()
        .reset_index()
    )
    pivot = grouped.pivot_table(
        index=["PARENT_DEALER_CODE_MODEL_FAMILY", "MONTH_DAY"],
        columns="YEAR",
        values="NET_SALES",
    )
    for y in hist_years:
        if y not in pivot.columns:
            pivot[y] = np.nan
    return pivot[hist_years]


def vectorized_linear_forecast(pivot: pd.DataFrame, hist_years, predict_year: int) -> pd.DataFrame:
    """Closed-form OLS across ALL (series, day) rows at once -- fast even at millions of rows."""
    Y = pivot.to_numpy(dtype=float)
    x = np.array(hist_years, dtype=float)

    mask = ~np.isnan(Y)
    n = mask.sum(axis=1)

    Y_filled = np.where(mask, Y, 0.0)
    X_filled = np.where(mask, np.broadcast_to(x, Y.shape), 0.0)

    sum_x = X_filled.sum(axis=1)
    sum_y = Y_filled.sum(axis=1)
    sum_xy = (X_filled * Y_filled).sum(axis=1)
    sum_x2 = (X_filled * X_filled).sum(axis=1)

    denom = n * sum_x2 - sum_x ** 2

    slope = np.zeros(len(pivot))
    intercept = np.zeros(len(pivot))

    # n >= 2 and not degenerate -> standard closed-form OLS
    valid = (n >= 2) & (denom != 0)
    slope[valid] = (n[valid] * sum_xy[valid] - sum_x[valid] * sum_y[valid]) / denom[valid]
    intercept[valid] = (sum_y[valid] - slope[valid] * sum_x[valid]) / n[valid]

    # exactly 1 point -> flat forecast = that value
    one_point = n == 1
    single_val = np.divide(sum_y, n, out=np.zeros_like(sum_y), where=n > 0)
    intercept[one_point] = single_val[one_point]
    slope[one_point] = 0.0

    # 0 points -> NaN
    no_point = n == 0

    prediction = slope * predict_year + intercept
    prediction[no_point] = np.nan
    prediction = np.where(np.isnan(prediction), prediction, np.clip(prediction, 0, None))

    result = pivot.reset_index()[["PARENT_DEALER_CODE_MODEL_FAMILY", "MONTH_DAY"]].copy()
    result["PREDICTED_NET_SALES"] = prediction
    result["N_HISTORICAL_POINTS"] = n.astype(int)
    return result


def finalize_output(result: pd.DataFrame, predict_year: int, window_start: str, window_end: str) -> pd.DataFrame:
    """Turn MONTH_DAY back into a real 2026 date, filter to the forecast window, tidy columns."""
    result = result.copy()
    result["CAL_DATE_2026"] = pd.to_datetime(
        f"{predict_year}-" + result["MONTH_DAY"], format="%Y-%m-%d", errors="coerce"
    )

    start = pd.to_datetime(f"{predict_year}-{window_start}", format="%Y-%m-%d")
    end = pd.to_datetime(f"{predict_year}-{window_end}", format="%Y-%m-%d")
    result = result[(result["CAL_DATE_2026"] >= start) & (result["CAL_DATE_2026"] <= end)]

    result["CAL_DATE"] = result["CAL_DATE_2026"].dt.strftime("%d-%m-%Y")

    out = result[
        ["PARENT_DEALER_CODE_MODEL_FAMILY", "CAL_DATE", "PREDICTED_NET_SALES", "N_HISTORICAL_POINTS"]
    ].sort_values(["PARENT_DEALER_CODE_MODEL_FAMILY", "CAL_DATE"])

    return out.reset_index(drop=True)

## 3. Main pipeline — takes the Snowpark DataFrame `df` in, returns a pandas DataFrame out

In [ ]:
def forecast_from_snowpark_df(snowpark_df, hist_years=HIST_YEARS, predict_year=PREDICT_YEAR,
                               window_start=WINDOW_START, window_end=WINDOW_END) -> pd.DataFrame:
    print("[1/4] Pulling Snowpark DataFrame into pandas ...")
    pdf = snowpark_df.to_pandas()
    pdf = normalize_columns(pdf)
    print(f"      -> {len(pdf):,} rows, {pdf['PARENT_DEALER_CODE_MODEL_FAMILY'].nunique():,} unique series")

    print("[2/4] Parsing dates and building (series x month-day) x year matrix ...")
    pdf = prep_dates(pdf)
    pivot = build_year_matrix(pdf, hist_years)
    print(f"      -> {len(pivot):,} (series, day) groups to fit")

    print("[3/4] Fitting vectorized linear regression for every group ...")
    result = vectorized_linear_forecast(pivot, hist_years, predict_year)

    print("[4/4] Formatting output ...")
    out = finalize_output(result, predict_year, window_start, window_end)

    n_no_history = out["PREDICTED_NET_SALES"].isna().sum()
    print(f"      -> Done: {len(out):,} predictions.")
    if n_no_history:
        print(f"      -> NOTE: {n_no_history:,} rows had zero historical data points and are NaN.")

    return out

## 4. Run it

In [ ]:
# `df` is your existing Snowpark DataFrame with columns
# PARENT_DEALER_CODE_MODEL_FAMILY, CAL_DATE, NET_SALES
predictions_pdf = forecast_from_snowpark_df(df)
predictions_pdf.head(20)

## 5. (Optional) Write predictions back to Snowflake

If you want the result back as a Snowpark DataFrame / Snowflake table rather than just a local
pandas DataFrame, uncomment and run this (requires an active `session`):

In [ ]:
# predictions_sdf = session.create_dataframe(predictions_pdf)
# predictions_sdf.write.mode("overwrite").save_as_table("YOUR_SCHEMA.NET_SALES_PREDICTIONS_2026")